# Fetal Planes Verification

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skimage
import skimage.io as io

RAW_DATA_DIR = Path("/scratch/data/fetal_planes_raw_20260519")

In [ ]:
# Check the `FETAL_PLANES_DB_data.csv` and `FETAL_PLANES_DB_data.xlsx` files.
# Are they the same?

csv_df = pd.read_csv(RAW_DATA_DIR / "FETAL_PLANES_DB_data.csv", delimiter=";")
xlsx_df = pd.read_excel(RAW_DATA_DIR / "FETAL_PLANES_DB_data.xlsx")

display(csv_df.head())  # ty:ignore[unresolved-reference]
display(xlsx_df.head())  # ty:ignore[unresolved-reference]

In [ ]:
# Print the unique values for the `Plane`, `Brain_plane`, `Operator`,
# `US_Machine` and `Train ` columns
print("Plane:", csv_df["Plane"].unique())
print("Brain_plane:", csv_df["Brain_plane"].unique())
print("Operator:", csv_df["Operator"].unique())
print("US_Machine:", csv_df["US_Machine"].unique())
print("Train:", csv_df["Train "].unique())

In [ ]:
# Check that there is an file {Image_name}.png for each row of the CSV data
# frame
for image_name in csv_df["Image_name"]:
    image_path = RAW_DATA_DIR / "Images" / f"{image_name}.png"
    if not image_path.exists():
        print(f"Image file {image_path} does not exist.")

In [ ]:
# Count the number of images and patients in the train and test sets
train_df = csv_df[csv_df["Train "] == 1]
test_df = csv_df[csv_df["Train "] == 0]

print("Train set:")
print(f"Number of images: {len(train_df)}")
print(f"Number of patients: {train_df['Patient_num'].nunique()}")

print("Test set:")
print(f"Number of images: {len(test_df)}")
print(f"Number of patients: {test_df['Patient_num'].nunique()}")

In [ ]:
# Count the number of patients and images for each class (`Plane`) in each
# split
for name, df in [("Train set", train_df), ("Test set", test_df)]:
    print(f"{name}:")
    class_counts = df.groupby("Plane").agg(
        num_patients=pd.NamedAgg(column="Patient_num", aggfunc="nunique"),
        num_images=pd.NamedAgg(column="Image_name", aggfunc="count"),
    )

    print("Total images:", len(df))
    print("Total patients:", df["Patient_num"].nunique())
    print(class_counts)
    print()

In [ ]:
# Load the images using scikit-image and report:
#   - number of images with N channels
#   - min/max pixel values
#   - min/mean/std/max height and width

images_with_n_channels = {}
min_pixel_value = float("inf")
max_pixel_value = float("-inf")
heights = []
widths = []

for filename in csv_df["Image_name"]:
    image_path = RAW_DATA_DIR / "Images" / f"{filename}.png"
    image = io.imread(image_path)

    # Update channel count
    n_channels = image.shape[2] if len(image.shape) == 3 else 1
    images_with_n_channels[n_channels] = images_with_n_channels.get(n_channels, 0) + 1

    # Update pixel value range
    min_pixel_value = min(min_pixel_value, image.min())
    max_pixel_value = max(max_pixel_value, image.max())

    # Update height and width lists
    heights.append(image.shape[0])
    widths.append(image.shape[1])

print("Number of images with N channels:", images_with_n_channels)
print("Min pixel value:", min_pixel_value)
print("Max pixel value:", max_pixel_value)
print("Height:")
print("\tMin:", np.min(heights))
print("\tMean ± std:", np.mean(heights), "±", np.std(heights))
print("\tMax:", np.max(heights))
print("Width:")
print("\tMin:", np.min(widths))
print("\tMean ± std:", np.mean(widths), "±", np.std(widths))
print("\tMax:", np.max(widths))

In [ ]:
# Create a new column `label` that is the same as `Plane`, but "Fetal brain"
# images are split into separate classes by `Brain_plane`
csv_df["label"] = csv_df.apply(
    lambda row: (
        f"{row['Plane']}_{row['Brain_plane']}"
        if row["Plane"] == "Fetal brain"
        else row["Plane"]
    ),
    axis=1,
)

csv_df.tail()

In [ ]:
# Count the number of patients and images for each new class (`Label`)
label_counts = csv_df.groupby("label").agg(
    num_patients=pd.NamedAgg(column="Patient_num", aggfunc="nunique"),
    num_images=pd.NamedAgg(column="Image_name", aggfunc="count"),
)

print("Total images:", len(csv_df))
print("Total patients:", csv_df["Patient_num"].nunique())
print(label_counts)

In [ ]:
# Visualize 5 images from each class in a 5 x N_CLASSES grid

labels = csv_df["label"].unique()
n_classes = len(labels)
fig, axes = plt.subplots(5, n_classes, figsize=(4 * n_classes, 20))

for i, label in enumerate(labels):
    label_images = csv_df[csv_df["label"] == label]["Image_name"].values
    for j in range(5):
        if j < len(label_images):
            image_path = RAW_DATA_DIR / "Images" / f"{label_images[j]}.png"
            image = io.imread(image_path)
            axes[j, i].imshow(image, cmap="gray")
            axes[j, i].set_title(f"{label} - {label_images[j]}")
        axes[j, i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Try to segment the scan region from the background using morphological
# operations. Plot the original image and the segmented mask for 5 samples
# from each class.


def segment_scan(image: np.ndarray) -> np.ndarray:
    """Generate a scan mask using morphological operations.

    Args:
        image: The input 2D image array.

    Returns:
        A binary mask of the scan region.
    """
    image = image.mean(axis=2) if len(image.shape) == 3 else image
    mask = image > 0

    mask = skimage.morphology.remove_small_objects(mask, min_size=5000)
    for _ in range(5):
        mask = skimage.morphology.binary_erosion(mask)
    for _ in range(5):
        mask = skimage.morphology.binary_dilation(mask)
    mask = skimage.morphology.convex_hull_image(mask)

    return mask.astype(np.uint8)


fig, axes = plt.subplots(5, n_classes, figsize=(4 * n_classes, 20))

for i, label in enumerate(labels):
    label_images = csv_df[csv_df["label"] == label]["Image_name"].values
    for j in range(5):
        if j < len(label_images):
            image_path = RAW_DATA_DIR / "Images" / f"{label_images[j]}.png"
            image = io.imread(image_path)
            mask = segment_scan(image)

            axes[j, i].imshow(image, cmap="gray", alpha=0.5)
            axes[j, i].imshow(mask, alpha=0.5, vmin=0, vmax=1)
            axes[j, i].set_title(
                f"{label} - {label_images[j]} - {image.min()}..{image.max()}"
            )
        axes[j, i].axis("off")
plt.tight_layout()
plt.show()